<a href="https://colab.research.google.com/github/dhag/colab_demo/blob/main/HTTP_flask_browser_ngrok.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HTTP 体験デモ（Flask サーバ・ブラウザから利用） / ngrok 公開

ブラウザで公開 URL を開くと**アップロードフォーム**が表示されます。
画像を選んで送信 → サーバが処理 → 結果がブラウザに表示・ダウンロードできます。
クライアントは普通のブラウザなので、別途ノートブックは不要です。

## 流れ
1. ブラウザが `GET /` → サーバが HTML フォームを返す。
2. フォーム送信で `POST /process`（multipart/form-data でファイル添付）。
3. サーバが処理し、結果画像を埋め込んだ HTML を返す。

処理本体（`--- 処理 ---`）はグレースケール化のサンプル。差し替えれば任意の処理にできます。

> 事前準備：ngrok 無料アカウントの authtoken（https://dashboard.ngrok.com/get-started/your-authtoken ）。
実行は上から順に。最後に出る URL を**ブラウザで開きます。**

## ① 依存パッケージをインストール

In [ ]:
!pip install -q flask pyngrok pillow

## ② Flask アプリを定義（HTML フォーム＋処理）

In [ ]:
import io, base64
from flask import Flask, request
from PIL import Image

app = Flask(__name__)

STYLE = """
body{font-family:system-ui,-apple-system,"Hiragino Kaku Gothic ProN",sans-serif;
     max-width:680px;margin:40px auto;padding:0 16px;color:#1a1a1a;line-height:1.6}
h1{font-size:1.3rem}
.card{border:1px solid #ddd;border-radius:10px;padding:20px;margin:16px 0}
button{padding:8px 18px;border:0;border-radius:8px;background:#2563eb;color:#fff;
       font-size:1rem;cursor:pointer}
input[type=file]{margin-bottom:12px}
img{max-width:100%;border-radius:8px;margin-top:12px}
a{color:#2563eb}
.meta{color:#666;font-size:.9rem}
"""

def page(title, body):
    return ("<!doctype html><html lang='ja'><head><meta charset='utf-8'>"
            "<meta name='viewport' content='width=device-width,initial-scale=1'>"
            "<title>" + title + "</title><style>" + STYLE + "</style></head>"
            "<body>" + body + "</body></html>")

@app.route("/")
def index():
    body = """
      <h1>画像アップロード → グレースケール処理</h1>
      <div class="card">
        <form action="/process" method="post" enctype="multipart/form-data">
          <input type="file" name="file" accept="image/*" required><br>
          <button type="submit">処理する</button>
        </form>
      </div>
      <p class="meta">画像を選んで送信すると、サーバ側でグレースケール化して返します。</p>
    """
    return page("画像処理デモ", body)

@app.route("/process", methods=["POST"])
def process():
    if "file" not in request.files or request.files["file"].filename == "":
        return page("エラー", "<p>ファイルが選択されていません。</p><p><a href='/'>← 戻る</a></p>"), 400
    f = request.files["file"]
    raw = f.read()

    # --- 処理本体（差し替え可）: 画像をグレースケール化 ---
    try:
        img = Image.open(io.BytesIO(raw)).convert("L")
    except Exception as e:
        return page("エラー", f"<p>画像として読めません: {e}</p><p><a href='/'>← 戻る</a></p>"), 400
    out = io.BytesIO(); img.save(out, "PNG"); result = out.getvalue()
    # ------------------------------------------------------

    b64 = base64.b64encode(result).decode()
    body = f"""
      <h1>処理結果</h1>
      <div class="card">
        <p class="meta">{f.filename} : {len(raw)} B → {len(result)} B</p>
        <img src="data:image/png;base64,{b64}" alt="result">
        <p><a href="data:image/png;base64,{b64}" download="gray_{f.filename}.png">ダウンロード</a></p>
      </div>
      <p><a href="/">← 別の画像を処理</a></p>
    """
    return page("処理結果", body)

## ③ Flask をバックグラウンドで起動

In [ ]:
import threading
PORT = 5000
def run():
    app.run(host="0.0.0.0", port=PORT, use_reloader=False, threaded=True)
threading.Thread(target=run, daemon=True).start()
print(f"Flask running on :{PORT}")

Flask running on :5000


## ④ ngrok で公開
authtoken 入力後に出る **URL をブラウザで開く**。

In [ ]:
import getpass
from pyngrok import ngrok, conf
print("ngrok authtoken を貼り付け（https://dashboard.ngrok.com/get-started/your-authtoken ）")
conf.get_default().auth_token = getpass.getpass()

public_url = ngrok.connect(PORT).public_url
print("=" * 50)
print("ブラウザでこの URL を開く:", public_url)
print("=" * 50)

ngrok authtoken を貼り付け（https://dashboard.ngrok.com/get-started/your-authtoken ）
··········
ブラウザでこの URL を開く: https://577a-34-181-179-0.ngrok-free.app


---
### メモ
- **ngrok 無料枠の警告ページ**：ブラウザで初回アクセスすると ngrok の確認ページが出る。
  「Visit Site」を押せば本来のフォームに進む（同じブラウザでは以後出ない）。これは無料枠の仕様。
- 処理本体を差し替えれば、リサイズ・二値化・エッジ抽出など任意の処理に変えられる。
- `<form ... enctype="multipart/form-data">` がファイル送信の要。`name="file"` が `request.files["file"]` に対応。
- スマホのブラウザからでも同じ URL でアクセスできる（授業で学生の端末から試させやすい）。